**Objective**
Train and evaluate baseline classifiers for each dataset independently to establish benchmark performance for downstream AI Trust Score computation.

Inputs
data/processed/

feature_driver_drowsiness.csv
feature_driver_distraction.csv
feature_fairface.csv
feature_celeba.csv

Outputs:
outputs/

models/
metrics/
predictions/

In [15]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt

In [19]:
from pathlib import Path

for f in Path("../data/processed").glob("*"):
    print(f.name)

cleaned_metadata.csv
feature_dataset.csv
master_metadata.csv
feature_celeba.csv
feature_driver_drowsiness.csv
feature_fairface.csv
feature_driver_distraction.csv


In [21]:
DATA = pd.read_csv(
    "../data/processed/feature_driver_drowsiness.csv"
)

print(DATA.shape)

DATA.head()

(1000, 15)


,file_name,size_mb,width,height,aspect_ratio,brightness,contrast,blur_score,image_area,size_per_pixel,is_square,Image_id,Left_eye_react,Right_eye_react,Label
0,driver_drowsiness_00000.jpg,0.034813,512,512,1.0,103.005287,74.732528,256.511483,262144,1.328008e-07,1,39367,"[292.0, 119.0, 30.0, 6.0]","[221.0, 124.0, 30.0, 7.0]",closed_eyes
1,driver_drowsiness_00001.jpg,0.032403,512,512,1.0,95.845665,58.721976,136.014380,262144,1.236076e-07,1,25965,"[276.0, 188.0, 44.0, 11.0]","[172.0, 187.0, 45.0, 11.0]",closed_eyes
2,driver_drowsiness_00002.jpg,0.033669,512,512,1.0,97.397602,67.644119,180.609741,262144,1.284352e-07,1,56838,"[271.0, 212.0, 43.0, 8.0]","[170.0, 211.0, 43.0, 8.0]",closed_eyes
3,driver_drowsiness_00003.jpg,0.032869,512,512,1.0,82.849628,53.563289,347.272139,262144,1.253866e-07,1,102894,"[330.0, 181.0, 44.0, 21.0]","[217.0, 176.0, 54.0, 21.0]",open_eyes
4,driver_drowsiness_00004.jpg,0.032885,512,512,1.0,82.899673,52.262713,157.250826,262144,1.254448e-07,1,32409,"[274.0, 158.0, 32.0, 7.0]","[203.0, 163.0, 28.0, 7.0]",closed_eyes


In [23]:
print(DATA["Label"].value_counts())

Label
closed_eyes    575
open_eyes      425
Name: count, dtype: int64


In [26]:
FEATURES = [

    "brightness",

    "contrast",

    "blur_score",

    "width",

    "height",

    "aspect_ratio",

    "image_area",

    "size_mb",

    "size_per_pixel",

    "is_square"

]

In [27]:
encoder = LabelEncoder()

DATA["target"] = encoder.fit_transform(DATA["Label"])

X = DATA[FEATURES]

y = DATA["target"]

print(X.shape)

(1000, 10)


In [28]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print(X_train.shape)

(800, 10)


In [29]:
MODELS = {

    "Random Forest":

        RandomForestClassifier(

            n_estimators=200,

            random_state=42

        ),

    "XGBoost":

        XGBClassifier(

            random_state=42,

            eval_metric="logloss"

        ),

    "LightGBM":

        LGBMClassifier(

            random_state=42

        )

}

In [30]:
OUTPUT_MODEL = Path("../outputs/models")
OUTPUT_MODEL.mkdir(parents=True, exist_ok=True)

OUTPUT_METRICS = Path("../outputs/metrics")
OUTPUT_METRICS.mkdir(parents=True, exist_ok=True)

OUTPUT_PRED = Path("../outputs/predictions")
OUTPUT_PRED.mkdir(parents=True, exist_ok=True)

results = []

In [31]:
for model_name, model in MODELS.items():

    print("="*60)

    print(model_name)

    model.fit(

        X_train,

        y_train

    )

    pred = model.predict(X_test)

    probability = model.predict_proba(X_test)

    accuracy = accuracy_score(

        y_test,

        pred

    )

    precision = precision_score(

        y_test,

        pred,

        average="weighted"

    )

    recall = recall_score(

        y_test,

        pred,

        average="weighted"

    )

    f1 = f1_score(

        y_test,

        pred,

        average="weighted"

    )

    results.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1

    })

    prediction_df = pd.DataFrame({

        "Actual": encoder.inverse_transform(y_test),

        "Prediction": encoder.inverse_transform(pred),

        "Probability": probability.max(axis=1)

    })

    prediction_df.to_csv(

        OUTPUT_PRED /
        f"{model_name}_predictions.csv",

        index=False

    )

    joblib.dump(

        model,

        OUTPUT_MODEL /
        f"{model_name}.pkl"

    )

    cm = confusion_matrix(

        y_test,

        pred

    )

    disp = ConfusionMatrixDisplay(

        confusion_matrix=cm,

        display_labels=encoder.classes_

    )

    disp.plot()

    plt.title(model_name)

    plt.savefig(

        OUTPUT_METRICS /
        f"{model_name}_confusion_matrix.png",

        dpi=300

    )

    plt.close()

    print(classification_report(

        y_test,

        pred

    ))

Random Forest
              precision    recall  f1-score   support

           0       0.65      0.75      0.69       115
           1       0.57      0.45      0.50        85

    accuracy                           0.62       200
   macro avg       0.61      0.60      0.60       200
weighted avg       0.61      0.62      0.61       200

XGBoost
              precision    recall  f1-score   support

           0       0.61      0.65      0.63       115
           1       0.48      0.44      0.46        85

    accuracy                           0.56       200
   macro avg       0.55      0.54      0.54       200
weighted avg       0.55      0.56      0.56       200

LightGBM
[LightGBM] [Info] Number of positive: 340, number of negative: 460
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000427 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train

In [32]:
results = pd.DataFrame(results)

results.sort_values(

    "Accuracy",

    ascending=False,

    inplace=True

)

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.62,0.612849,0.62,0.611290
1,XGBoost,0.56,0.554831,0.56,0.556531
2,LightGBM,0.55,0.550000,0.55,0.550000


In [33]:
results.to_csv(

    OUTPUT_METRICS /
    "baseline_model_results.csv",

    index=False

)

print(results)

           Model  Accuracy  Precision  Recall  F1 Score
0  Random Forest      0.62   0.612849    0.62  0.611290
1        XGBoost      0.56   0.554831    0.56  0.556531
2       LightGBM      0.55   0.550000    0.55  0.550000
